# 3PT Make Funnel Analysis

**Question:** Why are we only getting ~191 triggers out of ~9,700 3PT makes?

**Method:** Start from every ESPN 3PT make → trace forward to the next DK signal for that player → classify the dropout reason at each gate.

**Hypothesis:** Most makes are dropped because:
1. No DK signal at all for that player in that time window (player not tracked)
2. Signal exists but `bet_side = OVER` (MC model sees OVER value after line jumped) — these were silently excluded
3. Signal exists but line didn't jump (book didn't react)
4. Signal exists, line jumped, but poll interval was > 65s

In [ ]:
import subprocess, warnings, time, re
import duckdb, requests, pytz
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

warnings.filterwarnings('ignore')
ET = pytz.timezone('US/Eastern')
SESSION = requests.Session()
SESSION.verify = False
SESSION.headers.update({'User-Agent': 'Mozilla/5.0'})

BOOKMAKER         = 'draftkings'
WINDOW_SECONDS    = 60
NEXT_POLL_MAX     = 130   # look up to 130s after the make for the next signal
MIN_LINE_JUMP     = 0.5
MAX_POLL_INTERVAL = 65

S3_BUCKET  = 'nba-betting-mt'
S3_SIGNALS = 'data/04_output/live_betting_signals/player_points'

def _get_cred(k):
    r = subprocess.run(['aws','configure','get',k], capture_output=True, text=True, timeout=5)
    return r.stdout.strip()

AK, SK = _get_cred('aws_access_key_id'), _get_cred('aws_secret_access_key')

def duckdb_con():
    con = duckdb.connect(':memory:')
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.execute("SET s3_region='us-east-2';")
    con.execute(f"SET s3_access_key_id='{AK}';")
    con.execute(f"SET s3_secret_access_key='{SK}';")
    return con

print('Setup OK.')

## 1. Load ALL DraftKings signals (both OVER and UNDER)

In [ ]:
con = duckdb_con()
all_sigs = con.execute(f"""
    SELECT game_id, player_name, bookmaker, bet_side,
           save_timestamp_utc, current_points, live_line,
           over_odds, under_odds, edge_after
    FROM read_parquet('s3://{S3_BUCKET}/{S3_SIGNALS}/*.parquet')
    WHERE bookmaker = '{BOOKMAKER}'
      AND (bookmaker_stale IS NULL OR bookmaker_stale = FALSE)
""").fetchdf()
con.close()

all_sigs['ts'] = pd.to_datetime(all_sigs['save_timestamp_utc'], utc=True)
all_sigs = all_sigs.sort_values(['game_id','player_name','ts']).reset_index(drop=True)

# Prior poll within same (game, player)
grp = all_sigs.groupby(['game_id','player_name'])
all_sigs['prev_ts']       = grp['ts'].shift(1)
all_sigs['prev_line']     = grp['live_line'].shift(1)
all_sigs['prev_pts']      = grp['current_points'].shift(1)
all_sigs['line_delta']    = all_sigs['live_line'] - all_sigs['prev_line']
all_sigs['pts_delta']     = all_sigs['current_points'] - all_sigs['prev_pts']
all_sigs['poll_interval'] = (all_sigs['ts'] - all_sigs['prev_ts']).dt.total_seconds()

print(f'Total DK signals loaded: {len(all_sigs):,}')
print(f'bet_side breakdown:')
print(all_sigs['bet_side'].value_counts().to_string())
print(f'\nUnique games:   {all_sigs["game_id"].nunique():,}')
print(f'Unique players: {all_sigs["player_name"].nunique():,}')

## 2. Fetch ESPN 3PT Makes for All Games

In [ ]:
def normalize_name(name):
    return re.sub(r'[^a-z ]', '', name.lower()).strip()

def names_match(a, b):
    a, b = normalize_name(a), normalize_name(b)
    return a == b or a in b or b in a

def fetch_espn(game_id):
    url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}'
    try:
        r = SESSION.get(url, timeout=15); r.raise_for_status()
    except Exception:
        return [], {}
    data = r.json()
    # 3PT makes
    makes = []
    for p in data.get('plays', []):
        text = p.get('text', '')
        wc   = p.get('wallclock', '')
        if 'makes' not in text.lower(): continue
        if not any(x in text.lower() for x in ('three point','3-pt','three pointer')): continue
        if not wc: continue
        try: ts = datetime.fromisoformat(wc.replace('Z','+00:00'))
        except Exception: continue
        makes.append({
            'shooter': text.split(' makes ')[0].strip().lower(),
            'ts_utc':  ts,
            'period':  p.get('period',{}).get('number',0),
            'clock':   p.get('clock',{}).get('displayValue',''),
            'text':    text,
        })
    # Final scores
    scores = {}
    for team in data.get('boxscore',{}).get('players',[]):
        for sb in team.get('statistics',[]):
            labels = sb.get('labels',[])
            try: pts_idx = labels.index('PTS')
            except ValueError: continue
            for ath in sb.get('athletes',[]):
                name = ath.get('athlete',{}).get('displayName','').lower()
                stats_ = ath.get('stats',[])
                if pts_idx < len(stats_):
                    try: scores[name] = int(stats_[pts_idx])
                    except (ValueError, TypeError): pass
    return makes, scores

print('Fetch function defined.')

In [ ]:
game_ids = all_sigs['game_id'].astype(str).unique().tolist()
print(f'Fetching ESPN for {len(game_ids)} games...')

pbp_cache    = {}
scores_cache = {}

for i, gid in enumerate(game_ids):
    makes, scores = fetch_espn(gid)
    pbp_cache[gid]    = makes
    scores_cache[gid] = scores
    if (i+1) % 50 == 0 or (i+1) == len(game_ids):
        print(f'  {i+1}/{len(game_ids)} done')
    time.sleep(0.15)

total_makes = sum(len(v) for v in pbp_cache.values())
print(f'\nTotal 3PT makes cached: {total_makes:,}')
print(f'Avg per game: {total_makes/len(game_ids):.1f}')

## 3. Funnel: For Every 3PT Make, Trace to the Next Signal

For each 3PT make, find the next DK signal for the same player within `NEXT_POLL_MAX` seconds.
Classify why it does or doesn't become a trigger.

In [ ]:
# Build a fast lookup: for each (game_id, player_name), sorted signals
sig_lookup = {}
for (gid, pname), grp_df in all_sigs.groupby(['game_id','player_name']):
    sig_lookup[(str(gid), pname)] = grp_df.sort_values('ts').reset_index(drop=True)

def find_next_signal(game_id, shooter_lower, make_ts):
    """Find next signal row within NEXT_POLL_MAX seconds after a 3PT make."""
    # Match shooter to player_name in our signals
    gid = str(game_id)
    for (g, pname), sigs in sig_lookup.items():
        if g != gid: continue
        if not names_match(shooter_lower, pname): continue
        # Find first signal after make_ts
        after = sigs[sigs['ts'] > make_ts]
        if len(after) == 0: continue
        next_sig = after.iloc[0]
        gap = (next_sig['ts'] - make_ts).total_seconds()
        if gap <= NEXT_POLL_MAX:
            return next_sig, gap
        return None, gap  # signal exists but too late
    return None, None  # no signal for this player at all

print('Lookup built. Running funnel analysis...')
print(f'(This processes {total_makes:,} 3PT makes — may take a minute)')

In [ ]:
funnel_rows = []

for gid, makes in pbp_cache.items():
    for m in makes:
        shooter = m['shooter']
        make_ts = m['ts_utc']

        next_sig, gap = find_next_signal(gid, shooter, make_ts)

        row = {
            'game_id':  gid,
            'shooter':  shooter,
            'make_ts':  make_ts,
            'period':   m['period'],
            'clock':    m['clock'],
            'text':     m['text'],
            'gap_s':    gap,
        }

        if next_sig is None and gap is None:
            row['stage'] = 'NO_SIGNAL_PLAYER'       # player not in our tracking set
            row['bet_side'] = None
            row['line_delta'] = None
            row['poll_interval'] = None
        elif next_sig is None:
            row['stage'] = 'SIGNAL_TOO_LATE'         # next signal > 130s later
            row['bet_side'] = None
            row['line_delta'] = None
            row['poll_interval'] = None
        else:
            row['bet_side']      = next_sig['bet_side']
            row['line_delta']    = next_sig['line_delta']
            row['poll_interval'] = next_sig['poll_interval']
            row['live_line']     = next_sig['live_line']
            row['current_pts']   = next_sig['current_points']
            row['over_odds']     = next_sig['over_odds']
            row['under_odds']    = next_sig['under_odds']
            row['edge_after']    = next_sig['edge_after']

            # Get final score for outcome
            scores = scores_cache.get(gid, {})
            final_pts = next(
                (pts for name, pts in scores.items() if names_match(shooter, name)), None)
            row['final_pts'] = final_pts
            if final_pts is not None:
                row['over_outcome']  = 'WIN' if final_pts > next_sig['live_line'] else ('LOSS' if final_pts < next_sig['live_line'] else 'PUSH')
                row['under_outcome'] = 'WIN' if final_pts < next_sig['live_line'] else ('LOSS' if final_pts > next_sig['live_line'] else 'PUSH')
            else:
                row['over_outcome'] = row['under_outcome'] = None

            # Classify stage
            li = next_sig['line_delta']
            pi = next_sig['poll_interval']
            line_jumped    = (not pd.isna(li)) and (li >= MIN_LINE_JUMP)
            interval_clean = (not pd.isna(pi)) and (pi <= MAX_POLL_INTERVAL)

            if not interval_clean:
                row['stage'] = 'INTERVAL_TOO_LONG'
            elif not line_jumped:
                row['stage'] = 'LINE_NO_JUMP'
            else:
                row['stage'] = 'TRIGGER'             # would have fired

        funnel_rows.append(row)

funnel = pd.DataFrame(funnel_rows)
print(f'Funnel built: {len(funnel):,} rows (one per 3PT make)')

## 4. Funnel Breakdown

In [ ]:
print('=' * 60)
print('  3PT MAKE → TRIGGER FUNNEL')
print('=' * 60)
stage_counts = funnel['stage'].value_counts()
total = len(funnel)
for stage, n in stage_counts.items():
    print(f'  {stage:<25} {n:>5,}  ({n/total:.1%})')
print(f'  {"TOTAL":<25} {total:>5,}')

print()
print('Interpretation:')
print('  NO_SIGNAL_PLAYER  → player hits 3PT but we have no DK signal for them')
print('  SIGNAL_TOO_LATE   → next signal comes >130s after the make')
print('  INTERVAL_TOO_LONG → signal exists but prior poll was >65s ago')
print('  LINE_NO_JUMP      → signal exists, clean interval, but line did not jump ≥0.5')
print('  TRIGGER           → would fire (ESPN 3PT + clean interval + line jumped)')

In [ ]:
# Of TRIGGER rows: what bet_side does the model recommend?
triggers = funnel[funnel['stage'] == 'TRIGGER'].copy()
print(f'\nOf {len(triggers)} trigger-eligible makes:')
print(triggers['bet_side'].value_counts().to_string())
print()
print('>>> This is the KEY number: how many OVER vs UNDER after a 3PT that moved the line')

# OVER ROI if we bet OVER on every trigger
def american_to_decimal(odds):
    o = float(odds)
    return o/100 + 1 if o > 0 else 100/abs(o) + 1

over_evald = triggers[triggers['over_outcome'].isin(['WIN','LOSS'])].copy()
over_evald['decimal_over'] = over_evald['over_odds'].apply(american_to_decimal)
over_evald['over_profit']  = over_evald.apply(
    lambda r: (r['decimal_over']-1)*100 if r['over_outcome']=='WIN' else -100, axis=1)

n_ = len(over_evald)
w_ = (over_evald['over_outcome']=='WIN').sum()
roi_ = over_evald['over_profit'].sum() / (n_*100) if n_ else 0
print(f'\nAll trigger-eligible makes (regardless of bet_side):')
print(f'  n (with outcome):  {n_:,}')
print(f'  OVER W-L:          {w_}–{n_-w_}')
print(f'  OVER hit rate:     {w_/n_:.1%}' if n_ else '')
print(f'  OVER ROI:          {roi_:+.1%}')

In [ ]:
# Why does bet_side matter? Compare OVER ROI for UNDER-recommended vs OVER-recommended
print('=== OVER outcome by model bet_side recommendation ===')
for side in ['UNDER', 'OVER']:
    sub = over_evald[over_evald['bet_side'] == side]
    if len(sub) == 0: continue
    n_, w_ = len(sub), (sub['over_outcome']=='WIN').sum()
    r_ = sub['over_profit'].sum() / (n_*100)
    print(f'  Model said {side}: n={n_}  OVER hit={w_/n_:.1%}  ROI={r_:+.1%}')

print()
print('=== LINE_NO_JUMP: how often does a 3PT NOT move the line? ===')
no_jump = funnel[funnel['stage'] == 'LINE_NO_JUMP']
print(f'  {len(no_jump):,} makes ({len(no_jump)/total:.1%}) had a signal but no line jump')
print(f'  Mean line_delta: {no_jump["line_delta"].mean():.2f}')
print(f'  Distribution: {no_jump["line_delta"].describe(percentiles=[.25,.5,.75]).to_dict()}')

print()
print('=== NO_SIGNAL_PLAYER: how many unique players are we missing? ===')
no_signal = funnel[funnel['stage'] == 'NO_SIGNAL_PLAYER']
missing_shooters = no_signal['shooter'].nunique()
tracked_players  = all_sigs['player_name'].nunique()
print(f'  Unique shooters with no signal: {missing_shooters:,}')
print(f'  Unique players we DO track:     {tracked_players:,}')
print(f'  Top missing shooters (by 3PT make count):')
print(no_signal['shooter'].value_counts().head(20).to_string())

In [ ]:
# Full OVER ROI if we bet every trigger (OVER+UNDER) — no model filter
print('=== If we bet OVER on ALL trigger-eligible makes (any model recommendation) ===')
for period in sorted(over_evald['period'].dropna().unique()):
    sub = over_evald[over_evald['period']==period]
    n_, w_ = len(sub), (sub['over_outcome']=='WIN').sum()
    r_ = sub['over_profit'].sum()/(n_*100) if n_ else 0
    from scipy import stats as sc
    z = sc.norm.ppf(0.95)
    p = w_/n_; denom = 1+z**2/n_
    center = (p+z**2/(2*n_))/denom
    margin = z*np.sqrt(p*(1-p)/n_+z**2/(4*n_**2))/denom
    print(f'  Q{int(period)}: n={n_:3d}  OVER hit={w_/n_:.1%}  ROI={r_:+.1%}  CI=[{center-margin:.1%},{center+margin:.1%}]')

print()
# Line jump magnitude breakdown
jcuts = [0,1.0,2.0,3.0,float('inf')]
jlabels = ['+0.5–1.0','+1.0–2.0','+2.0–3.0','+3.0+']
over_evald['jump_bucket'] = pd.cut(over_evald['line_delta'], bins=jcuts, labels=jlabels)
print('By line jump magnitude:')
for bkt in jlabels:
    sub = over_evald[over_evald['jump_bucket']==bkt]
    if len(sub)==0: continue
    n_, w_ = len(sub), (sub['over_outcome']=='WIN').sum()
    r_ = sub['over_profit'].sum()/(n_*100)
    print(f'  {bkt}: n={n_:3d}  OVER hit={w_/n_:.1%}  ROI={r_:+.1%}')